# IR Sensor Library Documentation (IRSens)

## Overview
The **IRSens** library provides a simple and robust interface for working with infrared (IR) sensors on Raspberry Pi. This library handles GPIO configuration, sensor reading, and proper cleanup procedures for obstacle detection and line following applications.

**Full source code available at [IRSens.py](IRSens.py)**

## Features
- **Simple GPIO Management**: Automatic GPIO mode setup and pin configuration
- **Inverted Logic Handling**: Converts hardware logic to intuitive boolean values  
- **Debug Support**: Optional debug messages for troubleshooting
- **Error Handling**: Robust error detection and reporting
- **Proper Cleanup**: Safe GPIO cleanup to prevent conflicts
- **Singleton Pattern**: Prevents multiple GPIO mode initializations

## Libraries Used In The Script
- **RPi.GPIO**: GPIO control for Raspberry Pi
- **time**: Time delays and timing functions

## Implementation Guide

### 1. Import Required Libraries
First, we import the necessary libraries for GPIO control and timing functions:

In [ ]:
import RPi.GPIO as GPIO 
import time 

## 3. Class: `IRsens`

This class manages initialization, reading, and cleanup of an IR obstacle detection sensor connected to a Raspberry Pi.

- **Class Name**: `IRsens`  
- **Purpose**: Provide a simple interface for:
  - Initializing the GPIO pin for the IR sensor  
  - Reading sensor values (obstacle detection)  
  - Cleaning up GPIO resources  
  - Optional debug outputs for troubleshooting  



In [ ]:

__version__ = "1.0.4"

class IRsens:


### 3.1 The Constructor `__init__(self, IrPin=24, debug=False)`

This method initializes the IR sensor instance when the class is created.

- **Function Name**: `__init__`

- **Parameters**:
  - `IrPin` *(int, default=24)* → GPIO pin number connected to the IR sensor  
  - `debug` *(bool, default=False)* → Enable debug output  

- **Usage**:
  - Stores the IR pin and debug flag for later use  
  - Ensures that `GPIO.setmode(GPIO.BCM)` is called only once in the entire program  
  - Sets an internal flag `__init_check` to mark that initialization is complete  
  - If `debug=True`, prints a confirmation message:
    ```text
    IR Sensor Initialized
    ```


In [ ]:
  def __init__(self, IrPin = 24, debug=False): 
        self.Irsensor = IrPin
        self.debug = debug 
        if not IRsens.__init_check: 
            GPIO.setmode(GPIO.BCM)
            IRsens.__init_check = True
        if debug == True:
            print("IR Sensor Initialized")  
            

### 3.2 The `status(self)` Method  
This method reads the current state of the IR sensor and returns a normalized value.  

#### Method Structure:  
- **Input Pin Configuration**: Configure the IR sensor pin as input  
- **Read Sensor Value**: Read the digital signal from the IR sensor  
- **Normalize Output**: Convert raw input into `0` (clear) or `1` (obstacle)  
- **Debug Option**: Print sensor readings if `debug=True`  
- **Error Handling**: Raise an exception if reading fails  

#### Execution Steps:  
1. **Configure Input Pin**  
    ```python
    GPIO.setup(self.Irsensor, GPIO.IN)
    ```

2. **Read Sensor Value**  
    - If `GPIO.input(self.Irsensor) == 1` → set `stat = 0` *(no obstacle)*  
    - If `GPIO.input(self.Irsensor) == 0` → set `stat = 1` *(obstacle detected)*  

3. **Check for Invalid Reading**  
    - If `stat` remains `None`, log an error in debug mode and raise:  
    ```python
    raise ValueError("Error reading IR sensor")
    ```

4. **Debug Output** *(optional)*  
    ```python
    if self.debug:
        print(f"IR Sensor: {stat}")
    ```

5. **Return Value**  
    ```python
    return stat
    ```

#### Return Value:  
- **Type**: `int`  
- **Values**:  
  - `0` → No obstacle detected  
  - `1` → Obstacle detected  


In [ ]:
def status(self): 
        GPIO.setup(self.Irsensor, GPIO.IN )
        stat = None
        if GPIO.input(self.Irsensor) == 1: 
            stat = 0 
        elif GPIO.input(self.Irsensor) == 0: 
            stat = 1 
        if stat == None:
            if self.debug == True:
                print("IR Sensor: Error reading sensor")
            raise ValueError("Error reading IR sensor")    
        if self.debug == True:
            print(f"IR Sensor: {stat}")
        return stat
    

### 3.3 The Cleanup Method `cleanup(self)`

This method releases the GPIO resources used by the IR sensor.

- **Function Name**: `cleanup()`

- **Parameters**:  
  - None

- **Usage**:  
  - If `debug=True`, prints a confirmation message:  
    ```text
    IR Sensor Cleanup
    ```  
  - Calls `GPIO.cleanup(self.Irsensor)` to free the GPIO pin resources  
  - Handles exceptions during cleanup and prints the error if one occurs  
  - Resets the internal flag `__init_check` to `False`, allowing the sensor to be reinitialized later


In [ ]:
    
    def cleanup(self):
        if self.debug == True:
            print("IR Sensor Cleanup")
        try:
            GPIO.cleanup(self.Irsensor)
        except Exception as e:
            print(f"GPIO cleanup error: {e}")
        IRsens.__init_check = False
            

## 4. Example Usage

The following example shows how to use the `IRsens` class in a standalone script.



In [ ]:
if __name__ == "__main__":
    IR = IRsens(debug = True)
    try:
        while True: 
            print(IR.status())
            time.sleep(0.5)
    except KeyboardInterrupt:
        IR.cleanup()
        print("Program stopped by User")
    except Exception as e:
        print(f"An error occurred: {e}")
        IR.cleanup()